In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
from IPython.display import Image, display

display(Image("/kaggle/input/batch-normalization-notes-image/BN imge1.jpeg"))


In [ ]:
display(Image("/kaggle/input/batch-normalization-notes-image/BN image2.jpeg"))

In [ ]:
df = pd.read_csv("/kaggle/input/datasets/muhammadshahidazeem/customer-churn-dataset/customer_churn_dataset-training-master.csv")
df.head()

In [ ]:
df.shape

In [ ]:
df["Contract Length"].value_counts()

In [ ]:
df["Subscription Type"].value_counts()

In [ ]:
df["Gender"].value_counts()

## **Checking for dataset is balaced or not**

In [ ]:
df["Churn"].value_counts()

## **Converting object datatype columns into int datatype by using pd.get_dummies()**

In [ ]:
new_df = pd.get_dummies(df, dtype=int, drop_first=True)
new_df.head()

In [ ]:
new_df.shape

In [ ]:
new_df.info()

## **Checking for null values**

In [ ]:
new_df.isnull().sum()

## **Removing null values**

In [ ]:
new_df.dropna(inplace=True)

In [ ]:
new_df.isnull().sum()

## **Cheching for Duplicates records**

In [ ]:
new_df.duplicated().sum()

## **Removing CustomerID column because it is not contributing any thing to our analysis**

In [ ]:
new_df.drop("CustomerID", axis=1, inplace=True)

In [ ]:
new_df.head()

## **Checking for outliers and removing them from data**

In [ ]:
n = 1
plt.figure(figsize=(20,15))
for cols in new_df:
    if n <=7:
        ax=plt.subplot(3,3,n)
        plt.boxplot(new_df[cols])
        plt.title(cols)
    n += 1
plt.show()

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, precision_score, recall_score, f1_score
from sklearn.preprocessing import StandardScaler

import tensorflow
from tensorflow import keras
from keras import layers
from keras.models import Sequential
from keras.optimizers import Adam

## **Splitting data into train and test**

In [ ]:
x = new_df.drop("Churn", axis=1)
y = new_df["Churn"]

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(x,y, test_size = 0.3, random_state=42)

## **Standardizing x_train and x_test**

In [ ]:
scaler = StandardScaler()
x_train_scaled = scaler.fit_transform(x_train)
x_test_scaled = scaler.transform(x_test)

## **Without Batch Normalization**

In [ ]:
model = Sequential([
    layers.Dense(units=64,
                activation="relu",
                kernel_initializer = "he_normal",
                bias_initializer = "zeros",
                input_shape = [x_train.shape[1]]),
    layers.Dropout(0.2),

    layers.Dense(units=32,
                activation = "relu",
                kernel_initializer = "he_normal",
                bias_initializer = "zeros"),
    layers.Dropout(0.2),

    layers.Dense(units=16,
                activation = "relu",
                kernel_initializer = "he_normal",
                bias_initializer = "zeros"),
    layers.Dropout(0.2),

    layers.Dense(units=1,
                activation = "sigmoid",
                kernel_initializer = "glorot_normal")
])


#-------------------------------------------------------------------------------------------------------------------

model.compile(
    optimizer = "Adam",
    loss = "binary_crossentropy",
    metrics = ["f1_score"]
)

#---------------------------------------------------------------------------------------------------------------------

hist = model.fit(
    x_train_scaled, y_train,
    epochs = 20,
    batch_size = 128,
    validation_data = (x_test_scaled, y_test)
)

In [ ]:
pred = model.predict(x_test_scaled)
y_pred = np.where(pred>0.5,1,0)
print("Recall_score: ", recall_score(y_test, y_pred))
print()
print("precision_score: ", precision_score(y_test, y_pred))
print()
print("F1_score: ", f1_score(y_test, y_pred))
print()
print("Confusion Matrix: \n", confusion_matrix(y_test, y_pred))

In [ ]:
sns.heatmap(confusion_matrix(y_test, y_pred), 
            annot=True, fmt="d",
           xticklabels=[1,0],
           yticklabels=[1,0])

plt.xlabel("Predicted Values")
plt.ylabel("Actual Values")

plt.show()

In [ ]:
sns.heatmap(confusion_matrix(y_test, y_pred)/np.sum(confusion_matrix(y_test, y_pred)), 
            annot=True, fmt=".5%",
           xticklabels=[1,0],
           yticklabels=[1,0])

plt.xlabel("Predicted Values")
plt.ylabel("Actual Values")

plt.show()

In [ ]:
plt.plot(hist.history["loss"], label="loss")
plt.plot(hist.history["val_loss"], label="val_loss")
plt.legend()
plt.show()

In [ ]:
plt.plot(hist.history["f1_score"], label="F1")
plt.plot(hist.history["val_f1_score"], label="Val_F1")
plt.legend()
plt.show()

## **With Bacth Normalization**

In [ ]:
model_BN = Sequential([
    layers.Dense(units=64,
                activation="relu",
                kernel_initializer = "he_normal",
                bias_initializer = "zeros",
                input_shape = [x_train.shape[1]]),
    layers.BatchNormalization(),
    layers.Dropout(0.2),

    layers.Dense(units=32,
                activation = "relu",
                kernel_initializer = "he_normal",
                bias_initializer = "zeros"),
    layers.BatchNormalization(),
    layers.Dropout(0.2),

    layers.Dense(units=16,
                activation = "relu",
                kernel_initializer = "he_normal",
                bias_initializer = "zeros"),
    layers.BatchNormalization(),
    layers.Dropout(0.2),

    layers.Dense(units=1,
                activation = "sigmoid",
                kernel_initializer = "glorot_normal")
])


#-------------------------------------------------------------------------------------------------------------------

model_BN.compile(
    optimizer = "Adam",
    loss = "binary_crossentropy",
    metrics = ["f1_score"]
)

#---------------------------------------------------------------------------------------------------------------------

hist_BN = model_BN.fit(
    x_train_scaled, y_train,
    epochs = 20,
    batch_size = 128,
    validation_data = (x_test_scaled, y_test)
)

In [ ]:
pred = model_BN.predict(x_test_scaled)
BN_pred = np.where(pred>0.5,1,0)
print("Recall_score: ", recall_score(y_test, BN_pred))
print()
print("precision_score: ", precision_score(y_test, BN_pred))
print()
print("F1_score: ", f1_score(y_test, BN_pred))
print()
print("Confusion Matrix: \n", confusion_matrix(y_test, BN_pred))

In [ ]:
sns.heatmap(confusion_matrix(y_test, BN_pred), 
            annot=True, fmt="d",
           xticklabels=[1,0],
           yticklabels=[1,0])

plt.xlabel("Predicted Values")
plt.ylabel("Actual Values")

plt.show()

In [ ]:
plt.plot(hist_BN.history["f1_score"], label="F1_score")
plt.plot(hist_BN.history["val_f1_score"], label="Val_F1_score")
plt.legend()
plt.show()

In [ ]:
plt.plot(hist_BN.history["loss"], label="loss")
plt.plot(hist_BN.history["val_loss"], label="Val_loss")
plt.legend()
plt.show()